# Revised PCBert-Kla released-data benchmark

This notebook compares PCBert-Kla, MM-Kla, and architecture-aligned HybridKla retraining on the **same locked PCBert-Kla rows**. Contradictory labels and exact duplicates are removed. Candidate protein matches and qualifying full-protein similarity are grouped before splitting. The original rows do not provide verified parent-protein IDs, so this remains a retrospective internal diagnostic, not proof of protein-independent or external-study generalization. The HybridKla run uses its eight-branch model with grouped out-of-fold stacking, not the authors' published training protocol. Training is disabled by default.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, tempfile, urllib.request, zipfile

REPO_URL = 'https://github.com/kaleabnega/MM-Kla.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('/kaggle/working/MM-Kla')
PROJECT = REPO_DIR / 'experiment' / 'pcbert_kla_clean'
OUTPUT = Path('/kaggle/working/pcbert_revised_results')
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_PAIRED_MODELS = False
RUN_HYBRIDKLA_PILOT = False
RUN_HYBRIDKLA_FULL = False
LOCAL_HYBRIDKLA_ARCHIVE = None
SEEDS = (42, 123, 777)
EPOCHS = 30
PATIENCE = 6


## Checkout and environment

The private repository is cloned with the Kaggle secret `GITHUB_TOKEN`. The checked-out commit and available GPUs are recorded with the results.


In [ ]:
if not REPO_DIR.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Kaggle secret GITHUB_TOKEN is missing')
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\\n" x-access-token ;;\n  *Password*) printf "%s\\n" "$MM_KLA_GITHUB_TOKEN" ;;\n  *) exit 1 ;;\nesac\n')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'MM_KLA_GITHUB_TOKEN': token})
        result = subprocess.run(['git', '-c', 'credential.helper=', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], env=env, capture_output=True, text=True)
        del token, env
    if result.returncode:
        raise RuntimeError('Private clone failed; check token access and branch')
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=REPO_DIR, text=True).strip()
if branch != REPO_BRANCH:
    raise RuntimeError(f'Expected {REPO_BRANCH}, found {branch}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=PROJECT, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('A Kaggle GPU runtime is required for training')
runtime = {'commit': commit, 'branch': branch, 'gpu': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], 'torch': torch.__version__}
(OUTPUT / 'runtime.json').write_text(json.dumps(runtime, indent=2) + '\n')
print(json.dumps(runtime, indent=2))


The cloned source package is registered after checkout so notebook-side imports resolve in a fresh Kaggle run.


In [ ]:
import importlib
package_root = PROJECT / 'src'
if not (package_root / 'pcbert_kla_clean' / '__init__.py').is_file():
    raise FileNotFoundError(f'Missing cloned source package: {package_root}')
sys.path.insert(0, str(package_root))
importlib.invalidate_caches()


## Rebuild and verify the lock

The source CSVs, candidate mapping, homology clusters, and search hits are pinned by SHA-256. Regeneration must reproduce the committed manifests byte-for-byte before any model is trained.


In [ ]:
committed = PROJECT / 'review_data' / 'pcbert_revised_v1'
regenerated = OUTPUT / 'regenerated_lock'
result = subprocess.run([sys.executable, 'scripts/lock_pcbert_revised.py', '--output-dir', str(regenerated)], cwd=PROJECT, text=True, capture_output=True)
if result.returncode:
    raise RuntimeError('Lock regeneration failed:\n' + result.stdout[-2000:] + result.stderr[-2000:])
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
for filename in ('full_manifest.csv', 'model_manifest.csv', 'split_audit.json'):
    if sha256(committed / filename) != sha256(regenerated / filename):
        raise ValueError('Regenerated lock differs from committed ' + filename)
lock = committed / 'split_audit.json'
audit = json.loads(lock.read_text())
check = subprocess.run([sys.executable, 'scripts/run_replication.py', '--run', 'pcbert-revised-check', '--pcbert-revised-lock', str(lock)], cwd=PROJECT, text=True, capture_output=True, check=True)
print(check.stdout)
print('Original test after exact train-overlap exclusion:', audit['original_test_if_train_overlaps_removed'])


## PCBert-Kla and MM-Kla training

Three predeclared seeds use the same locked train, validation, and internal-test rows for both architectures. Each job has a 30-epoch maximum with validation-AUPRC early stopping and a fixed 0.5 decision threshold. The internal test is not used to select settings. On two GPUs, the models run concurrently, one per device.


In [ ]:
def start_run(architecture, seed, device):
    destination = OUTPUT / f'seed_{seed}' / architecture
    destination.mkdir(parents=True, exist_ok=True)
    args = [sys.executable, 'scripts/run_replication.py', '--run', 'pcbert-revised', '--pcbert-revised-lock', str(lock), '--architecture', architecture, '--model-name', 'Rostlab/prot_bert', '--optimizer', 'sgd', '--learning-rate', '0.003', '--weight-decay', '0', '--scheduler', 'none', '--epochs', str(EPOCHS), '--patience', str(PATIENCE), '--batch-size', '4', '--device', device, '--seed', str(seed), '--output-dir', str(destination)]
    logfile = (destination / 'train.log').open('w')
    process = subprocess.Popen(args, cwd=PROJECT, stdout=logfile, stderr=subprocess.STDOUT)
    return process, logfile, destination

if RUN_PAIRED_MODELS:
    for seed in SEEDS:
        jobs = {}
        try:
            for architecture in ('baseline', 'token_gated'):
                device = 'cuda:0' if architecture == 'baseline' or torch.cuda.device_count() == 1 else 'cuda:1'
                jobs[architecture] = start_run(architecture, seed, device)
                if torch.cuda.device_count() == 1:
                    process, logfile, destination = jobs.pop(architecture)
                    code = process.wait()
                    logfile.close()
                    if code:
                        raise RuntimeError(architecture + ' failed; see ' + str(destination / 'train.log'))
            for architecture, (process, logfile, destination) in list(jobs.items()):
                code = process.wait()
                logfile.close()
                del jobs[architecture]
                if code:
                    raise RuntimeError(architecture + ' failed; see ' + str(destination / 'train.log'))
        finally:
            for process, logfile, destination in jobs.values():
                if process.poll() is None:
                    process.terminate()
                    process.wait()
                logfile.close()
        print('Completed matched seed', seed)
else:
    print('Matched training disabled')


## HybridKla architecture-aligned retraining

HybridKla's ESM2, LSTM, six sequence-derived descriptor branches, and meta-classifier use the same locked rows and fixed internal test as the two ProtBert models. Its descriptors are computed from the same windows, not from the released 27-feature table. Grouped out-of-fold stacking replaces the authors' row-wise protocol. The branch-specific maxima are 40 ESM2, 30 LSTM, 100 feature-DNN, and 50 meta-classifier epochs, each selected by validation AUPRC with patience 6. This is a matched-data comparison, not an equal-compute or exact published-model reproduction. The official feature-resource archive is verified by SHA-256. The optional pilot trains one ESM2 fold without test inference; the full run is substantially more expensive than either single ProtBert model.


In [ ]:
if RUN_HYBRIDKLA_PILOT or RUN_HYBRIDKLA_FULL:
    source = OUTPUT / 'hybridkla_source'
    source.mkdir(exist_ok=True)
    if LOCAL_HYBRIDKLA_ARCHIVE is None:
        archive_path = source / 'HybridKla.zip'
        if not archive_path.is_file():
            urllib.request.urlretrieve('https://transkla.zzu.edu.cn/api/download/HybridKla.zip', archive_path)
    else:
        archive_path = Path(LOCAL_HYBRIDKLA_ARCHIVE)
    if not archive_path.is_file():
        raise FileNotFoundError(archive_path)
    if sha256(archive_path) != '73ddb128c441d7b213c4636e6bb6ce34501e82143c2b9bc06ec329d68a422993':
        raise ValueError('HybridKla resource archive checksum mismatch')
    resources = source / 'HybridKla' / 'train'
    resources.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for filename in ('top10.txt', 'BLOSUM62R.txt'):
            (resources / filename).write_bytes(archive.read('HybridKla/train/' + filename))
    from pcbert_kla_clean.hybridkla_features import load_resources
    load_resources(resources / 'top10.txt', resources / 'BLOSUM62R.txt')
    def run_hybridkla(seed, pilot):
        destination = OUTPUT / f'seed_{seed}' / ('hybridkla_pilot' if pilot else 'hybridkla')
        destination.mkdir(parents=True, exist_ok=True)
        args = [sys.executable, 'scripts/run_hybridkla_architecture.py', '--benchmark-lock', str(lock), '--resource-dir', str(resources), '--output-dir', str(destination), '--device', 'cuda:0', '--seed', str(seed)]
        if pilot:
            args.append('--pilot')
        with (destination / 'train.log').open('w') as logfile:
            result = subprocess.run(args, cwd=PROJECT, stdout=logfile, stderr=subprocess.STDOUT)
        if result.returncode:
            raise RuntimeError(f'HybridKla failed for seed {seed}; see {destination / "train.log"}')
        report = json.loads((destination / 'run_report.json').read_text())
        if report['benchmark_lock_sha256'] != sha256(lock) or report['seed'] != seed:
            raise ValueError('HybridKla report does not match the locked seed and split')
        return report
    if RUN_HYBRIDKLA_PILOT:
        pilot_report = run_hybridkla(SEEDS[0], pilot=True)
        if pilot_report['test_metrics']:
            raise ValueError('Timing pilot unexpectedly scored the test set')
        print('HybridKla one-fold ESM2 pilot seconds:', pilot_report['wall_seconds'])
    if RUN_HYBRIDKLA_FULL:
        for seed in SEEDS:
            report = run_hybridkla(seed, pilot=False)
            print('HybridKla seed', seed, report['test_metrics']['internal_test'])
else:
    print('HybridKla retraining disabled')


## Generated comparison

The table is populated only from completed runs. All three methods use the same internal-test samples and the same 0.5 threshold. HybridKla uses architecture-specific optimization and grouped stacking; its score is not a published HybridKla result. No paper-reported metric is inserted into the table.


In [ ]:
import pandas as pd
records = []
for seed in SEEDS:
    for architecture in ('baseline', 'token_gated'):
        directory = OUTPUT / f'seed_{seed}' / architecture / 'pcbert_revised' / 'runs'
        if not directory.exists():
            continue
        files = sorted(directory.glob('*/test_metrics.json'))
        if len(files) != 1:
            raise RuntimeError(f'Expected one result for {architecture}, seed {seed}')
        config = json.loads(files[0].with_name('run_config.json').read_text())
        if config.get('revised_lock_sha256') != sha256(lock) or config.get('model_seed') != seed:
            raise ValueError('ProtBert result does not match the locked seed and split')
        records.append({'seed': seed, 'architecture': architecture, **json.loads(files[0].read_text())})
    report_path = OUTPUT / f'seed_{seed}' / 'hybridkla' / 'run_report.json'
    if report_path.is_file():
        report = json.loads(report_path.read_text())
        if report['benchmark_lock_sha256'] != sha256(lock) or report['seed'] != seed:
            raise ValueError('HybridKla result does not match the locked seed and split')
        metrics = report['test_metrics'].get('internal_test')
        if metrics is None:
            raise ValueError('HybridKla full run lacks internal-test metrics')
        records.append({'seed': seed, 'architecture': 'hybridkla_architecture_aligned', **metrics})
results = pd.DataFrame.from_records(records)
if results.empty:
    print('No completed model evaluations')
else:
    display(results)
    results.to_csv(OUTPUT / 'three_model_internal_test.csv', index=False)
    expected = {(seed, name) for seed in SEEDS for name in ('baseline', 'token_gated', 'hybridkla_architecture_aligned')}
    if set(zip(results['seed'], results['architecture'])) == expected and len(results) == len(expected):
        summary = results.groupby('architecture')[['AUPRC', 'AUC', 'MCC', 'F1', 'Pre', 'Rec', 'SP', 'ACC']].agg(['mean', 'std'])
        display(summary)
        summary.to_csv(OUTPUT / 'three_model_summary.csv')
    else:
        print('Three-model summary pending incomplete seeds or models')


## Internal-test uncertainty

Whole candidate-homology groups are resampled to give descriptive 95% intervals for AUPRC and AUC. These intervals do not make the retrospective test independent.


In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score
manifest = pd.read_csv(committed / 'model_manifest.csv')
test_manifest = manifest.loc[manifest['partition'] == 'internal_test'].set_index('window_id')
if len(test_manifest) != audit['partitions_before_training_resampling']['internal_test']['rows']:
    raise ValueError('Locked internal-test size changed')
intervals = []
for seed in SEEDS:
    prediction_paths = {}
    for architecture in ('baseline', 'token_gated'):
        directory = OUTPUT / f'seed_{seed}' / architecture / 'pcbert_revised' / 'runs'
        if directory.exists():
            files = sorted(directory.glob('*/diagnostic_test_predictions.csv'))
            if len(files) != 1:
                raise RuntimeError(f'Expected one prediction file for {architecture}, seed {seed}')
            prediction_paths[architecture] = files[0]
    hybrid_predictions = OUTPUT / f'seed_{seed}' / 'hybridkla' / 'internal_test_predictions.csv'
    if hybrid_predictions.is_file():
        prediction_paths['hybridkla_architecture_aligned'] = hybrid_predictions
    for architecture, path in prediction_paths.items():
        frame = pd.read_csv(path)
        if architecture == 'hybridkla_architecture_aligned':
            ids = frame['Site_ID'].astype(str)
            labels = frame['Label'].to_numpy(dtype=int)
            scores = frame['Score'].to_numpy(dtype=float)
        else:
            ids = frame['name'].astype(str)
            labels = frame['true_label'].to_numpy(dtype=int)
            scores = frame['score'].to_numpy(dtype=float)
        if len(ids) != len(test_manifest) or ids.duplicated().any() or set(ids) != set(test_manifest.index):
            raise ValueError(f'{architecture} predictions do not cover the locked test exactly')
        expected_labels = test_manifest.loc[ids, 'label'].to_numpy(dtype=int)
        if not np.array_equal(labels, expected_labels) or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
            raise ValueError(f'{architecture} labels or scores differ from the lock')
        groups = test_manifest.loc[ids, 'candidate_group'].to_numpy()
        unique = np.unique(groups)
        index = {group: np.flatnonzero(groups == group) for group in unique}
        rng = np.random.default_rng(2026)
        bootstrap = []
        for _ in range(2000):
            selected_groups = rng.choice(unique, size=len(unique), replace=True)
            selected = np.concatenate([index[group] for group in selected_groups])
            if len(np.unique(labels[selected])) < 2:
                continue
            bootstrap.append((average_precision_score(labels[selected], scores[selected]), roc_auc_score(labels[selected], scores[selected])))
        if len(bootstrap) < 1900:
            raise RuntimeError('Too many degenerate group-bootstrap samples')
        bounds = np.percentile(np.asarray(bootstrap), [2.5, 97.5], axis=0)
        intervals.append({'seed': seed, 'architecture': architecture, 'positive_test_windows': int(labels.sum()), 'AUPRC_low': bounds[0, 0], 'AUPRC_high': bounds[1, 0], 'AUC_low': bounds[0, 1], 'AUC_high': bounds[1, 1]})
if intervals:
    display(pd.DataFrame(intervals))
    pd.DataFrame(intervals).to_csv(OUTPUT / 'three_model_group_bootstrap_intervals.csv', index=False)
else:
    print('No completed prediction files')


## Scope

This diagnostic answers exact-window duplication and candidate-protein similarity at the declared threshold. Candidate accessions are sequence matches, **not verified original parent IDs**; the original released benchmark cannot establish a fully protein-independent or new-study test. The held-out test here is an internal repartition of old data. These rows were already examined during earlier model development, so the repartition is retrospective, not freshly blinded. External validation requires a separate accession-labelled source.
